### **Imports:**

In [1]:
import pandas as pd
import re
import itertools
import joblib
import requests
import json

from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, RandomForestRegressor, VotingRegressor, BaggingRegressor, StackingRegressor
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, make_scorer, root_mean_squared_error

from tqdm.notebook import tqdm
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor


### **Reading File**

In [2]:
df = pd.read_csv('data/epi_r.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20052 entries, 0 to 20051
Columns: 680 entries, title to turkey
dtypes: float64(679), str(1)
memory usage: 104.0 MB


In [3]:
df.describe()

,rating,calories,protein,fat,sodium,#cakeweek,#wasteless,22-minute meals,3-ingredient recipes,30 days of groceries,...,yellow squash,yogurt,yonkers,yuca,zucchini,cookbooks,leftovers,snack,snack week,turkey
count,20052.000000,1.593500e+04,15890.000000,1.586900e+04,1.593300e+04,20052.000000,20052.000000,20052.000000,20052.000000,20052.000000,...,20052.000000,20052.000000,20052.000000,20052.000000,20052.000000,20052.000000,20052.000000,20052.000000,20052.000000,20052.000000
mean,3.714467,6.322958e+03,100.160793,3.468775e+02,6.225975e+03,0.000299,0.000050,0.000848,0.001346,0.000349,...,0.001247,0.026332,0.000050,0.000299,0.014861,0.000150,0.000349,0.001396,0.000948,0.022741
std,1.340829,3.590460e+05,3840.318527,2.045611e+04,3.333182e+05,0.017296,0.007062,0.029105,0.036671,0.018681,...,0.035288,0.160123,0.007062,0.017296,0.121001,0.012231,0.018681,0.037343,0.030768,0.149080
min,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.750000,1.980000e+02,3.000000,7.000000e+00,8.000000e+01,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,4.375000,3.310000e+02,8.000000,1.700000e+01,2.940000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,4.375000,5.860000e+02,27.000000,3.300000e+01,7.110000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,5.000000,3.011122e+07,236489.000000,1.722763e+06,2.767511e+07,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### **Data preparation:**

In [4]:
df_title = df['title']
non_binary_cols = ['title', 'calories', 'protein', 'fat', 'sodium']

df.drop(columns=non_binary_cols, inplace=True)

In [5]:
df.drop(columns=['#cakeweek', '#wasteless', '22-minute meals', '3-ingredient recipes', '30 days of groceries',
                 'advance prep required', 'alabama', 'alaska', 'alcoholic', 'anniversary', 'anthony bourdain', 'aperitif', 'aperitif', 'arizona', 'aspen', 'atlanta', 'australia',
                 'back to school', 'backyard bbq', 'bastille day', 'beverly hills', 'birthday', 'blender', 'boil', 'bon appétit', 'bon app��tit', 'boston', 'braise', 'breakfast', 'broil', 'brooklyn', 'brunch', 'buffalo', 'buffet', 'bulgaria',
                 'california', 'cambridge', 'camping', 'canada', 'candy thermometer', 'chicago', 'chill', 'christmas', 'christmas eve', 'cocktail', 'cocktail party', 'coffee grinder', 'colorado', 'columbus', 'connecticut', 'cook like a diner', 'cookbook critic', 'costa mesa', 'cuba',
                 'dairy free', 'dallas', 'deep-fry', 'denver', 'dessert', 'digestif', 'dinner', 'diwali', 'dominican republic', 'dorie greenspan', 'double boiler', 'drink', 'drinks',
                 'easter', 'edible gift', 'egypt', 'emeril lagasse', 'engagement party', 'england', 'entertaining', 'epi + ushg', 'epi loves the microwave',
                 'fall', 'family reunion', 'fat free', "father's day", 'flaming hot summer', 'florida', 'food processor', 'fourth of july', 'france', 'frankenrecipe', 'freeze/chill', 'freezer food', 'friendsgiving', 'frozen dessert', 'fry', 'fruit',
                 'game', 'georgia', 'germany', 'gourmet', 'graduation', 'grill', 'grill/barbecue', 'guam',
                 'haiti', 'halloween', 'hanukkah', 'hawaii', 'healdsburg', 'healthy', 'high fiber', 'hollywood', "hors d'oeuvre", 'hot drink', 'house & garden', 'house cocktail', 'houston',
                 'ice cream machine', 'idaho', 'illinois', 'indiana', 'iowa', 'ireland', 'israel', 'italy',
                 'jamaica', 'japan', 'juicer',
                 'kansas', 'kansas city', 'kentucky', 'kentucky derby', 'kid-friendly', 'kidney friendly', 'kitchen olympics', 'kosher', 'kosher for passover',
                 'labor day', 'lancaster', 'las vegas', 'london', 'los angeles', 'louisiana', 'louisville', 'low cal', 'low carb', 'low cholesterol', 'low fat', 'low sodium', 'low sugar', 'low/no sugar', 'lunar new year', 'lunch',
                 'maine', 'maryland', 'massachusetts', 'mexico', 'miami', 'michigan', 'microwave', 'minneapolis', 'minnesota', 'mississippi', 'missouri', 'mixer', 'monterey jack', 'mortar and pestle', "mother's day",
                 'nancy silverton', 'nebraska', 'new hampshire', 'new jersey', 'new mexico', 'new orleans', "new year's day", "new year's eve", 'new york', 'no meat, no problem', 'no sugar added', 'no-cook', 'non-alcoholic', 'north carolina',
                 'ohio', 'oklahoma', 'oktoberfest', 'one-pot meal', 'oregon', 'organic',
                 'pan-fry', 'parade', 'paris', 'party', 'pasadena', 'passover', 'pasta maker', 'peanut free', 'pennsylvania', 'persian new year', 'peru', 'pescatarian', 'philippines', 'pittsburgh', 'pizza', 'poach', 'poker/game night', 'port', 'portland', 'pot pie', 'potluck', 'pressure cooker', 'providence', 'purim',
                 'quick & easy', 'quick and healthy',
                 'ramadan', 'raw', 'rhode island', 'roast', 'rosh hashanah/yom kippur', 'rub',
                 'san francisco', 'sandwich', 'sandwich theory', 'santa monica', 'seattle', 'self', 'shower', 'side', 'simmer', 'skewer', 'slow cooker', 'smoker', 'smoothie', 'soup/stew', 'south carolina', 'soy free', 'spain', 'spring', 'st. louis', "st. patrick's day", 'steam', 'stew', 'stir-fry', 'stock', 'stuffing/dressing', 'sugar conscious', 'summer', 'super bowl', 'suzanne goin', 'switzerland',
                 'tailgating', 'tennessee', 'tested & improved', 'texas', 'thanksgiving', 'tree nut free',
                 'utah', "valentine's day", 'vegan', 'vegetarian', 'vermont', 'virginia',
                 'washington', 'washington, d.c.', 'wedding', 'west virginia', 'wheat/gluten-free', 'whole wheat', 'windsor', 'winter', 'wisconsin', 'wok',
                 'cookbooks', 'leftovers', 'snack', 'snack week' ],
        inplace=True)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20052 entries, 0 to 20051
Columns: 404 entries, rating to turkey
dtypes: float64(404)
memory usage: 61.8 MB


In [11]:
df_title

0                    Lentil, Apple, and Turkey Wrap 
1        Boudin Blanc Terrine with Red Onion Confit 
2                      Potato and Fennel Soup Hodge 
3                   Mahi-Mahi in Tomato Olive Sauce 
4                          Spinach Noodle Casserole 
                            ...                     
20047                                Parmesan Puffs 
20048                Artichoke and Parmesan Risotto 
20049                         Turkey Cream Puff Pie 
20050       Snapper on Angel Hair with Citrus Cream 
20051    Baked Ham with Marmalade-Horseradish Glaze 
Name: title, Length: 20052, dtype: str

In [18]:
df1 = df.copy()
df1['title'] = df_title
df1.to_csv('data/epi_r_new.csv')

In [8]:
min_cols = df.columns[1:][df.iloc[:, 1:].mean() < 0.001]
max_cols = df.columns[1:][df.iloc[:, 1:].mean() > 0.2]

df.drop(columns=[*min_cols, *max_cols], inplace=True)

markers_to_drop = ['day', 'night', 'morning', 'summer', 'winter', 'mother', 'father', 'no', 'non', 'wedding',
                   'washington', 'texas', 'thanksgiving', 'sea', 'ramadan', 'quick', 'year', 'low', 'graduation',
                   'family', 'free', 'date', 'party', 'christmas', 'california', 'breakfast', 'school', 'anniversary',
                   '3-ingredient recipes', 'advance prep', 'required', 'alcohol', 'patrick', 'dinner', 'friendly',
                   'pennsylvania', 'dairy', 'dessert', 'buffet', 'backyard bbq', 'candy',
                   'game', 'fry', 'grill', 'picnic', 'pastry', 'michigan', 'massachusetts', 'mix', 'new', 'fest',
                   'stuff', 'fruit', 'leftovers', 'stew', 'braise', 'republic', 'fall', 'spring', 'high', 'lunch',
                   'boil', 'blender', 'snack', 'fourth of july', 'gift']

drop_columns = []
for marker in markers_to_drop:
    for column in df.columns[1:]:
        if marker in column:
            drop_columns.append(column)
df.drop(columns=drop_columns, inplace=True)

df.columns = (list(map(lambda column: re.split(r"\bor\b|/", column)[0].strip()
            if '/' in column
            or (len(column.split()) > 1 and column.split()[1] == 'or')
            else column, df.columns)))

df = df.loc[:, ~df.columns.duplicated()]

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20052 entries, 0 to 20051
Columns: 287 entries, rating to turkey
dtypes: float64(287)
memory usage: 43.9 MB


In [10]:
df

,rating,almond,amaretto,anchovy,anise,appetizer,apple,apricot,artichoke,arugula,...,wasabi,watercress,watermelon,weelicious,whiskey,white wine,wine,yogurt,zucchini,turkey
0,2.500,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,4.375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20047,3.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20048,4.375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
20049,4.375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
20050,4.375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### **Checking ingredient columns and rating for Nan values**

In [217]:
df.columns[df.isna().any()]

Index([], dtype='str')

**------------------------------------------------------------------------------------------------------------------------**


### **Classification Models Training and Scoring**

#### **Models to Train:**

In [98]:
models_to_train = [
    ('Random Forest', RandomForestClassifier(random_state=21), { #params
                                                                'n_estimators': [150, 200, 250],
                                                                'max_depth': [20, 30]
                                                                }),
    ('SVC', Pipeline([('scaler', StandardScaler()), ('svc', SVC(random_state=21, probability=True))]), { #params
                                                                                      'svc__C': [1, 10],
                                                                                      'svc__gamma': ['scale']
                                                                                      }),
    ('Gradient Boosting', GradientBoostingClassifier(random_state=21), { #params
                                                                        'learning_rate': [0.01],
                                                                        'n_estimators': [300],
                                                                        'max_depth': [6, 9],
                                                                        }),
    ('CatBoost', CatBoostClassifier(random_state=21, verbose=False), { #params
                                                                      'iterations': [300],
                                                                      'learning_rate': [0.01],
                                                                      'depth': [6, 9],
                                                                     }),
    ('XGBoost', XGBClassifier(random_state=21, n_jobs=-1, eval_metric=['mlogloss']), { #params
                                                                                     'learning_rate': [0.01],
                                                                                     'n_estimators': [300],
                                                                                     'max_depth': [6, 9],
                                                                                      })
]

#### **Binarizing to the closest integer, analyzing accuracies**

In [99]:
x = df.drop(columns='rating')
y = df['rating'].round().astype('int')

In [100]:
y.value_counts(normalize=True)

rating
4    0.657690
5    0.135597
0    0.091562
3    0.074257
2    0.032715
1    0.008179
Name: proportion, dtype: float64

In [101]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=21)

In [102]:
def dummy(x_train, x_test, y_train, y_test):
    dummy_clf = DummyClassifier(strategy='most_frequent')
    dummy_clf.fit(x_train, y_train)
    dummy_predicted = dummy_clf.predict(x_test)
    dummy_accuracy = accuracy_score(y_test, dummy_predicted)
    return dummy_accuracy

In [103]:
def b_estimators(models_to_train, scoring, scoring_f, x_train, x_test, y_train, y_test):

    pbar = tqdm(models_to_train)
    best_estimators = {}

    for model_name, model, params in pbar:

        pbar.set_description(f"Processing {model_name}")
        grid = GridSearchCV(
            model,
            params,
            cv=5,
            scoring=scoring,
            n_jobs=-1
        )

        grid.fit(x_train, y_train)

        if scoring_f == precision_score:
            scoring_func = scoring_f(y_test, grid.predict(x_test), average='weighted', labels=[2], zero_division=0)
        else:
            scoring_func = scoring_f(y_test, grid.predict(x_test))

        best_estimators[model_name] = (grid.best_estimator_, scoring_func)

    best_estimators = dict(sorted(best_estimators.items(), key=lambda x: x[1][1]))

    return best_estimators

In [104]:
dummy_accuracy = dummy(x_train, x_test, y_train, y_test)
best_estimators = b_estimators(models_to_train, 'accuracy', accuracy_score, x_train, x_test, y_train, y_test)

  0%|          | 0/5 [00:00<?, ?it/s]

In [105]:
def show_model_scores(best_estimators, dummy_accuracy):

    for model_name, (_, accuracy) in best_estimators.items():
        print('-' * 30)
        print(f"{model_name} accuracy on test dataset: {accuracy}")
        print(f"{model_name} accuracy compare to Naive accuracy: {accuracy} vs {dummy_accuracy}")
        print(f"{'-' * 30}\n")

    best_model = list(best_estimators)[-1]
    _, best_accuracy = best_estimators[best_model]

    print(f"Best Model: {best_model}\nAccuracy: {best_accuracy}")

show_model_scores(best_estimators, dummy_accuracy)

------------------------------
CatBoost accuracy on test dataset: 0.6679132385938669
CatBoost accuracy compare to Naive accuracy: 0.6679132385938669 vs 0.6576913487908252
------------------------------

------------------------------
Gradient Boosting accuracy on test dataset: 0.668910496135627
Gradient Boosting accuracy compare to Naive accuracy: 0.668910496135627 vs 0.6576913487908252
------------------------------

------------------------------
XGBoost accuracy on test dataset: 0.6724008975317876
XGBoost accuracy compare to Naive accuracy: 0.6724008975317876 vs 0.6576913487908252
------------------------------

------------------------------
SVC accuracy on test dataset: 0.6726502119172276
SVC accuracy compare to Naive accuracy: 0.6726502119172276 vs 0.6576913487908252
------------------------------

------------------------------
Random Forest accuracy on test dataset: 0.6771378708551483
Random Forest accuracy compare to Naive accuracy: 0.6771378708551483 vs 0.6576913487908252
---

#### **Binarizing to Categorical, analyzing accuracies and precisions**

##### **Accuracy on categorical target value**

In [106]:
def binarizing(x):
    if x <= 1:
        category = 0 #bad
    elif x <= 3:
        category = 1 #so-so
    else:
        category = 2 #great
    return category

y = y.apply(binarizing)

In [107]:
y.value_counts(normalize=True)

rating
2    0.793287
1    0.106972
0    0.099741
Name: proportion, dtype: float64

In [108]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=21)
dummy_accuracy = dummy(x_train, x_test, y_train, y_test)

In [109]:
best_cat_estimators = b_estimators(models_to_train, 'accuracy', accuracy_score, x_train, x_test, y_train, y_test)

  0%|          | 0/5 [00:00<?, ?it/s]

In [110]:
show_model_scores(best_cat_estimators, dummy_accuracy)

------------------------------
CatBoost accuracy on test dataset: 0.7960608327100474
CatBoost accuracy compare to Naive accuracy: 0.7960608327100474 vs 0.7933183744702069
------------------------------

------------------------------
XGBoost accuracy on test dataset: 0.8002991772625281
XGBoost accuracy compare to Naive accuracy: 0.8002991772625281 vs 0.7933183744702069
------------------------------

------------------------------
SVC accuracy on test dataset: 0.8022936923460484
SVC accuracy compare to Naive accuracy: 0.8022936923460484 vs 0.7933183744702069
------------------------------

------------------------------
Gradient Boosting accuracy on test dataset: 0.8022936923460484
Gradient Boosting accuracy compare to Naive accuracy: 0.8022936923460484 vs 0.7933183744702069
------------------------------

------------------------------
Random Forest accuracy on test dataset: 0.8047868362004488
Random Forest accuracy compare to Naive accuracy: 0.8047868362004488 vs 0.7933183744702069
-

##### **Voting Classifier accuracy on Categorical target value**

###### **We have chosen categorical tagret value to investigate, because it contains only 3 classes, which makes the classification task easier and faster. We will be using soft voting classifier, since it calculates probabilities of models predictions, which allows us to obtain more precisely evaluation metrics!**

In [111]:
estimators = [(model_name, estimator) for model_name, (estimator, _) in best_cat_estimators.items()]

In [112]:
voting_clf_acc = VotingClassifier(
    estimators=estimators,
    voting='soft'
)

voting_clf_acc.fit(x_train, y_train)
y_pred = voting_clf_acc.predict(x_test)
voting_clf_accuracy = accuracy_score(y_test, y_pred)
best_cat_estimators['Voting Classifier'] = (voting_clf_acc, voting_clf_accuracy)

print(f"Voting Classifier accuracy on categorical target value: {voting_clf_accuracy}")

Voting Classifier accuracy on categorical target value: 0.8017950635751683


##### **Precision on categorical target value (on great values)**

In [113]:
custom_precision = make_scorer(precision_score, average='weighted', labels=[2], zero_division=0)
best_cat_precision_estimators = b_estimators(models_to_train, custom_precision, precision_score, x_train, x_test, y_train, y_test)

  0%|          | 0/5 [00:00<?, ?it/s]

In [114]:
for model_name, (_, precision) in best_cat_precision_estimators.items():
        print('-' * 30)
        print(f"{model_name} precision on test dataset for great: {precision}")
        print(f"{'-' * 30}\n")


best_model = list(best_cat_precision_estimators)[-1]
_, best_accuracy = best_cat_precision_estimators[best_model]
print(f"Best Model: {best_model}\nPrecision: {best_accuracy}")

------------------------------
CatBoost precision on test dataset for great: 0.79768495218923
------------------------------

------------------------------
XGBoost precision on test dataset for great: 0.8018724696356275
------------------------------

------------------------------
Gradient Boosting precision on test dataset for great: 0.806245200921423
------------------------------

------------------------------
Random Forest precision on test dataset for great: 0.8083908928114608
------------------------------

------------------------------
SVC precision on test dataset for great: 0.8185516680227829
------------------------------

Best Model: SVC
Precision: 0.8185516680227829


##### **Voting Classifier precision on Categorical target value**

In [115]:
estimators = [(model_name, estimator) for model_name, (estimator, _) in best_cat_precision_estimators.items()]

In [116]:
voting_clf_prec = VotingClassifier(
    estimators=estimators,
    voting='soft'
)

voting_clf_prec.fit(x_train, y_train)
y_pred = voting_clf_prec.predict(x_test)
voting_clf_precision = precision_score(y_test, y_pred, average='weighted', labels=[2], zero_division=0)
best_cat_precision_estimators['Voting Classifier'] = (voting_clf_prec, voting_clf_precision)

print(f"Voting Classifier precision on categorical target value: {voting_clf_precision}")

Voting Classifier precision on categorical target value: 0.8033949835317963


### **Conclusion**

In [117]:
model_names = best_cat_estimators.keys()
acc_prec = []
for model_name in model_names:
    acc_prec.append((list(best_cat_estimators[model_name])[1],
                     list(best_cat_precision_estimators[model_name])[1]))

best_models = sorted(dict(zip(model_names, acc_prec)).items(), key=lambda x: (x[1][1], x[1][0]))

for model_name, (acc, prec) in best_models:
    print(f"{model_name}:\n\tAccuracy: {acc}\n\tPrecision: {prec}\n")

CatBoost:
	Accuracy: 0.7960608327100474
	Precision: 0.79768495218923

XGBoost:
	Accuracy: 0.8002991772625281
	Precision: 0.8018724696356275

Voting Classifier:
	Accuracy: 0.8017950635751683
	Precision: 0.8033949835317963

Gradient Boosting:
	Accuracy: 0.8022936923460484
	Precision: 0.806245200921423

Random Forest:
	Accuracy: 0.8047868362004488
	Precision: 0.8083908928114608

SVC:
	Accuracy: 0.8022936923460484
	Precision: 0.8185516680227829



### **The best and final classification model is SVC!**

In [118]:
svc_model = list(best_cat_precision_estimators['SVC'])[0]
joblib.dump(svc_model, 'data/best_classification_model.pkl')

['data/best_classification_model.pkl']

**------------------------------------------------------------------------------------------------------------------------**

### **Regression models Training and Scoring**

In [140]:
x_reg = df.drop(columns='rating')
y_reg = df['rating']

In [141]:
X_train, X_test, Y_train, Y_test = train_test_split(x_reg, y_reg, test_size=0.2, stratify=y, random_state=21)

In [150]:
n_jobs = -1
cv = 5

#### **Naive Regressor**

In [151]:
y_naive_pred = [Y_test.mean()] * len(Y_test)
naive_rmse = root_mean_squared_error(Y_test, y_naive_pred)
print(f"RMSE-score for naive regressor is {naive_rmse}")

RMSE-score for naive regressor is 1.345686213688248


#### **Linear Regressor**

In [163]:
lin_reg = LinearRegression()
lin_score = cross_validate(lin_reg, X_train, Y_train, return_train_score=True, cv=cv, scoring='neg_root_mean_squared_error')
print(f"Average RMSE on crossval is {-lin_score['test_score'].mean()}")

Average RMSE on crossval is 1.272238538442151


#### **Decision Tree Regressor**

In [162]:
tree_reg_params = {
    'max_depth' : [2, 3, 5, 6, 7, 10, 15, 20]
}
tree_reg = DecisionTreeRegressor(random_state=21)
tree_reg_gs = GridSearchCV(tree_reg, tree_reg_params, n_jobs=n_jobs, cv=cv, scoring='neg_root_mean_squared_error')
tree_reg_gs.fit(X_train, Y_train)

print(f"Best parameters are {tree_reg_gs.best_params_}")
print(f"Average RMSE on crossval for best parameters is {-tree_reg_gs.best_score_}")

Best parameters are {'max_depth': 6}
Average RMSE on crossval for best parameters is 1.299552762737197


#### **Random Forest Regressor**

In [154]:
rf_reg_params = {
    'max_depth' : [10, 20, 23, 40],
    'n_estimators' : [10, 25, 50, 75]
}
rf_reg = RandomForestRegressor(random_state = 21)
rf_reg_gs = GridSearchCV(rf_reg, rf_reg_params, n_jobs=n_jobs, cv=cv, scoring='neg_root_mean_squared_error')
rf_reg_gs.fit(X_train, Y_train)

print(f"Best parameters are {rf_reg_gs.best_params_}")
print(f"Average RMSE on crossval for best parameters is {-rf_reg_gs.best_score_}")

Best parameters are {'max_depth': 40, 'n_estimators': 75}
Average RMSE on crossval for best parameters is 1.2849476395013875


#### **The best regression estimator is RandomForest!**

In [171]:
best_reg_score = root_mean_squared_error(Y_test, rf_reg_gs.best_estimator_.predict(X_test))
print(f"Best RMSE-score on test subsample for best regression model is {best_reg_score}")

Best RMSE-score on test subsample for best regression model is 1.2693226708025396


#### **Regression Ensembles**

##### **Voting Regressor**

In [172]:
voting_reg_gs_params = {'weights' : list(itertools.product(
    range(1,6),
    range(1,6),
    range(1,6)))}
voting_reg = VotingRegressor(estimators=[('lin', lin_reg), ('tree', tree_reg_gs.best_estimator_), ('rf', rf_reg_gs.best_estimator_)])
voting_reg_gs = GridSearchCV(voting_reg, voting_reg_gs_params, n_jobs=n_jobs, cv=cv, scoring='neg_root_mean_squared_error')
voting_reg_gs.fit(X_train, Y_train)
print(f"Best parameters are {voting_reg_gs.best_params_}")
print(f"Average RMSE on crossval for best parameters is {-voting_reg_gs.best_score_}")

Best parameters are {'weights': (5, 1, 4)}
Average RMSE on crossval for best parameters is 1.258564084955909


##### **Bagging Regressor**

In [173]:
bagging_reg_gs_params = {
    'n_estimators' : [10, 20, 30, 50]
}
bagging_reg = BaggingRegressor(estimator = lin_reg, random_state = 21)
bagging_reg_gs = GridSearchCV(bagging_reg, bagging_reg_gs_params, n_jobs=n_jobs, cv=cv, scoring='neg_root_mean_squared_error')
bagging_reg_gs.fit(X_train, Y_train)
print(f"Best parameters are {bagging_reg_gs.best_params_}")
print(f"Average RMSE on crossval for best parameters is {-bagging_reg_gs.best_score_}")

Best parameters are {'n_estimators': 50}
Average RMSE on crossval for best parameters is 1.2726138084551566


##### **Stacking Regressor**

In [174]:
stacking_reg_gs_params = {
    'passthrough' : (True, False)
}
stacking_reg = StackingRegressor(estimators=[('lin', lin_reg), ('tree', tree_reg_gs.best_estimator_), ('rf', rf_reg_gs.best_estimator_)])
stacking_reg_gs = GridSearchCV(stacking_reg, stacking_reg_gs_params, n_jobs=n_jobs, cv=cv,scoring='neg_root_mean_squared_error')
stacking_reg_gs.fit(X_train, Y_train)
print(f"Best parameters are {stacking_reg_gs.best_params_}")
print(f"Average RMSE on crossval for best parameters is {-stacking_reg_gs.best_score_}")

Best parameters are {'passthrough': False}
Average RMSE on crossval for best parameters is 1.2584196457314265


##### **The best regression ensemble is Stacking!**

In [178]:
best_ensemble_score = root_mean_squared_error(Y_test, stacking_reg_gs.best_estimator_.predict(X_test))
print(f"Best RMSE-score on test subsample for best ensemble regression model is {best_ensemble_score}")

Best RMSE-score on test subsample for best ensemble regression model is 1.2526212165347497


### **Decision: it is better to use classification model for this kind of task**

**------------------------------------------------------------------------------------------------------------------------**


### **Nutrition Facts**

In [179]:
api_key = 'PGRM6CVqVJAO7UGSTTsELnZhptWbIU0crzwl4Pyh'
api_url = f"https://api.nal.usda.gov/fdc/v1/foods/search?api_key={api_key}&query="

In [180]:
rows = []
for col in x_reg.columns:
    row = {'title' : col}
    query = col.replace(' ', '%20').replace('/', '%20')
    response = requests.get(api_url + query)
    py_dict = json.loads(response.text)
    if py_dict['totalHits'] > 0:
        for nutrient in py_dict['foods'][0]['foodNutrients']:
            nutrientName = nutrient['nutrientName']
            value = nutrient['value']
            row[nutrientName] = value
    rows.append(row)

nutrition_facts_df = pd.DataFrame(rows)

In [182]:
nutrition_facts_df_pers = nutrition_facts_df['title'].to_frame()

In [184]:
nutrition_facts_df_pers['fat'] = nutrition_facts_df['Total lipid (fat)'] / 78 * 100
nutrition_facts_df_pers['Saturated fat'] = nutrition_facts_df['Fatty acids, total saturated'] / 20 * 100
nutrition_facts_df_pers['Cholesterol'] = nutrition_facts_df['Cholesterol'] / 300 * 100
nutrition_facts_df_pers['Total carbohydrates'] = nutrition_facts_df['Carbohydrate, by difference'] / 275 * 100
nutrition_facts_df_pers['Sodium'] = nutrition_facts_df['Sodium, Na'] / 2300 * 100
nutrition_facts_df_pers['Dietary Fiber'] = nutrition_facts_df['Fiber, total dietary'] / 28 * 100
nutrition_facts_df_pers['Protein'] = nutrition_facts_df['Protein'] / 50 * 100
nutrition_facts_df_pers['Added sugars'] = nutrition_facts_df['Sugars, added'] / 50 * 100
nutrition_facts_df_pers['Vitamin A'] = nutrition_facts_df['Vitamin A, RAE'] / 900 * 100
nutrition_facts_df_pers['Vitamin C'] = nutrition_facts_df['Vitamin C, total ascorbic acid'] / 90 * 100
nutrition_facts_df_pers['Calcium'] = nutrition_facts_df['Calcium, Ca'] / 1300 * 100
nutrition_facts_df_pers['Iron'] = nutrition_facts_df['Iron, Fe'] / 18 * 100
nutrition_facts_df_pers['Vitamin D'] = nutrition_facts_df['Vitamin D (D2 + D3)'] / 20 * 100
nutrition_facts_df_pers['Vitamin E'] = (nutrition_facts_df['Vitamin E (alpha-tocopherol)'] + nutrition_facts_df['Vitamin E, added']) / 15 * 100
nutrition_facts_df_pers['Vitamin K'] = (nutrition_facts_df['Vitamin K (phylloquinone)'] + nutrition_facts_df['Vitamin K (Dihydrophylloquinone)'] + nutrition_facts_df['Vitamin K (Menaquinone-4)']) / 120 * 100
nutrition_facts_df_pers['Thiamin'] = nutrition_facts_df['Thiamin'] / 1.2 * 100
nutrition_facts_df_pers['Riboflavin'] = nutrition_facts_df['Riboflavin'] / 1.3 * 100
nutrition_facts_df_pers['Niacin'] = nutrition_facts_df['Niacin'] / 16 * 100
nutrition_facts_df_pers['Vitamin B6'] = nutrition_facts_df['Vitamin B-6'] / 1.7 * 100
nutrition_facts_df_pers['Folate'] = nutrition_facts_df['Folate, DFE'] / 400 * 100
nutrition_facts_df_pers['Vitamin B12'] = nutrition_facts_df['Vitamin B-12'] / 2.4 * 100
nutrition_facts_df_pers['Biotin'] = nutrition_facts_df['Biotin'] / 30 * 100
nutrition_facts_df_pers['Pantothenic acid'] = nutrition_facts_df['Pantothenic acid'] / 5 * 100
nutrition_facts_df_pers['Phosphorus'] = nutrition_facts_df['Phosphorus, P'] / 1250 * 100
nutrition_facts_df_pers['Magnesium'] = nutrition_facts_df['Magnesium, Mg'] / 420 * 100
nutrition_facts_df_pers['Zinc'] = nutrition_facts_df['Zinc, Zn'] / 11 * 100
nutrition_facts_df_pers['Selenium'] = nutrition_facts_df['Selenium, Se'] / 55 * 100
nutrition_facts_df_pers['Copper'] = nutrition_facts_df['Copper, Cu'] / 0.9 * 100
nutrition_facts_df_pers['Manganese'] = nutrition_facts_df['Manganese, Mn'] / 2.3 * 100
nutrition_facts_df_pers['Molybdenum'] = nutrition_facts_df['Molybdenum, Mo'] / 45 * 100
nutrition_facts_df_pers['Potassium'] = nutrition_facts_df['Potassium, K'] / 4700 * 100
nutrition_facts_df_pers['Choline'] = nutrition_facts_df['Choline, total'] / 550 * 100

In [185]:
nutrition_facts_df_pers.fillna(0, inplace=True)

,title,fat,Saturated fat,Cholesterol,Total carbohydrates,Sodium,Dietary Fiber,Protein,Added sugars,Vitamin A,...,Pantothenic acid,Phosphorus,Magnesium,Zinc,Selenium,Copper,Manganese,Molybdenum,Potassium,Choline
0,almond,67.589744,21.125,0.000000,7.676364,10.086957,34.285714,41.34,0.0,0.000000,...,0.00,40.32,63.333333,28.727273,1.454545,107.666667,0.0,0.0,15.765957,9.418182
1,amaretto,12.820513,16.650,0.000000,12.109091,2.913043,0.000000,0.00,66.6,0.000000,...,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000
2,anchovy,12.448718,11.015,28.333333,0.000000,159.478261,0.000000,57.78,0.0,1.333333,...,0.00,20.16,16.428571,22.181818,123.818182,37.666667,0.0,0.0,11.574468,15.454545
3,anise,20.384615,2.930,0.000000,18.181818,0.695652,52.142857,35.20,0.0,1.777778,...,15.94,35.20,40.476190,48.181818,9.090909,101.111111,100.0,0.0,30.638298,0.000000
4,appetizer,5.128205,5.000,0.000000,3.272727,19.565217,10.714286,2.00,8.0,0.000000,...,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0,10.212766,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,white wine,0.000000,0.000,0.000000,0.945455,0.217391,0.000000,0.14,0.0,0.000000,...,0.00,1.44,2.380952,1.090909,0.181818,0.444444,0.0,0.0,1.510638,0.781818
282,wine,0.000000,0.000,0.000000,0.589091,0.478261,0.000000,0.08,0.0,0.000000,...,0.00,0.96,1.666667,1.090909,0.181818,0.777778,0.0,0.0,1.276596,0.545455
283,yogurt,4.512821,11.000,5.000000,1.603636,2.086957,0.000000,7.04,0.0,0.000000,...,0.00,0.00,3.095238,5.636364,0.000000,0.000000,0.0,0.0,3.276596,0.000000
284,zucchini,0.000000,0.000,0.000000,1.530909,0.000000,3.928571,2.10,0.0,0.000000,...,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.0,0.0,4.723404,0.000000


In [187]:
nutrition_facts_df_pers.to_csv("data/nutrition_facts.csv", index=False)

**------------------------------------------------------------------------------------------------------------------------**

### **Similar Recipes**

In [ ]:
epicurious_base_url = "https://epicurious.com"
epicurious_search_url = epicurious_base_url + "/search?q="
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9'
    }
session = requests.Session()
session.headers.update(headers)
rows = []

def parse_recipe(title):
    title = title.strip()
    url = None
    query = title.replace(",", "%2C").replace(" ", "+")
    response = session.get(epicurious_search_url + query)
    soup = BeautifulSoup(response.content, "html.parser")
    recipe_h2 = soup.find("h2", string = title)
    if recipe_h2 is not None:
        href = recipe_h2.parent['href']
        url = epicurious_base_url + href
        print(url)

    return {'title' : title, 'url' : url}


with ThreadPoolExecutor(max_workers=10) as executor:
    rows = list(executor.map(parse_recipe, df_title.tolist()))

epicurious_df = pd.DataFrame(rows).dropna(axis=0)

In [ ]:
epicurious_df.to_csv("data/similar_recipes_new.csv", index=False)